# Wordle GRPO — 完整 463 詞 held-out 評測

這本 notebook 只做推理，不重新訓練。請在 Colab 選 **L4 GPU（建議）**；若 L4 暫時不可用才選 A100。Notebook 會主動拒絕 T4/CPU，避免跑到一半才 OOM。

會在完全相同的 463 個 held-out 答案、greedy decoding 與固定協定下，比較 base 與公開 GRPO LoRA。兩者共用同一個 vLLM engine，結果與 log 會保存到 Drive。

**你只要做三件事：** (1) 在 Colab 開啟本檔；(2) `執行階段 → 變更執行階段類型 → L4 GPU`；(3) 按「全部執行」。

若 Drive 已有正確的 `MyDrive/agentic-rl-wordle/wordle_rl_bundle.zip`，流程會直接繼續；若缺少或版本錯誤，第二格會自動跳出檔案選擇器，你只要選本專案根目錄的 `wordle_rl_bundle.zip`。不用手動建立資料夾或改路徑。

**保守時間估計：L4 首次執行約 25–45 分鐘**（安裝/下載 10–20 分鐘，base + LoRA 完整 463 詞約 15–25 分鐘）。A100 約 15–30 分鐘，但較耗 compute units。

In [ ]:
# ① 固定參數：通常不必修改
DRIVE_BASE = "/content/drive/MyDrive/agentic-rl-wordle"
BUNDLE = f"{DRIVE_BASE}/wordle_rl_bundle.zip"
ADAPTER = "steven0226/qwen2.5-1.5b-wordle-grpo"
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
RESULT_DIR = f"{DRIVE_BASE}/evaluations/full-463"
BUNDLE_SHA256 = "79979fff55f4d3abe91cd75dc509d7b7b50a664ecfc51bbc9e32bb9324c3a4bf"
SEED = 42
N = 463
print("設定完成：L4、eval_full、463 words、greedy；不需修改任何參數")

In [ ]:
# ② 全自動 preflight：掛載 Drive、確認 GPU、取得並驗證正確 bundle
from google.colab import drive
drive.mount("/content/drive")

import hashlib
import pathlib
import shutil
import torch

assert torch.cuda.is_available(), "未偵測到 GPU：請到 執行階段 > 變更執行階段類型 > L4 GPU"
gpu = torch.cuda.get_device_properties(0)
gpu_gib = gpu.total_memory / 1024**3
print(f"GPU: {gpu.name} ({gpu_gib:.1f} GiB)")
gpu_name = gpu.name.upper()
assert ("L4" in gpu_name or "A100" in gpu_name) and gpu_gib >= 20, (
    f"目前是 {gpu.name} ({gpu_gib:.1f} GiB)。請改選 L4（建議）或 A100；不要用 T4。"
)
if "A100" in gpu_name:
    print("提示：A100 可以跑而且較快；若想省 compute units，L4 已足夠。")

free_gib = shutil.disk_usage("/content").free / 1024**3
assert free_gib >= 10, f"Colab 本機磁碟只剩 {free_gib:.1f} GiB，需要至少 10 GiB"
print(f"本機可用磁碟：{free_gib:.1f} GiB")

bundle_path = pathlib.Path(BUNDLE)
def bundle_is_current(path):
    return path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == BUNDLE_SHA256

if not bundle_is_current(bundle_path):
    print("Drive 缺少正確版本的 bundle。請在接下來的視窗選擇 wordle_rl_bundle.zip。")
    from google.colab import files
    uploaded = files.upload()
    data = uploaded.get("wordle_rl_bundle.zip")
    assert data is not None, "必須選擇檔名完全相同的 wordle_rl_bundle.zip"
    actual_sha256 = hashlib.sha256(data).hexdigest()
    assert actual_sha256 == BUNDLE_SHA256, (
        "你選到舊版 bundle。請回到筆電專案根目錄，選擇最新的 wordle_rl_bundle.zip。"
    )
    bundle_path.parent.mkdir(parents=True, exist_ok=True)
    bundle_path.write_bytes(data)
    print("正確 bundle 已自動保存到 Drive。")

assert bundle_is_current(bundle_path)
print("bundle SHA256 驗證通過")

!rm -rf /content/agentic-rl-wordle
!mkdir -p /content/agentic-rl-wordle
!unzip -q -o "$BUNDLE" -d /content/agentic-rl-wordle
%cd /content/agentic-rl-wordle
print("source bundle 解壓完成")

In [ ]:
# ③ 安裝已驗證的依賴；任何 pip 錯誤都會立刻停止，不會帶病進入評測
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"
], check=True)
subprocess.run([sys.executable, "scripts/fetch_words.py"], check=True)

import trl, vllm
print("trl", trl.__version__, "| vllm", vllm.__version__)
assert trl.__version__ == "1.8.0", f"TRL 版本錯誤：{trl.__version__}"
assert vllm.__version__ == "0.23.0", f"vLLM 版本錯誤：{vllm.__version__}"
print("依賴與 2,315 詞資料完成")

In [ ]:
# ④ 公開模型不強制需要 token；若 Colab Secrets 有 HF_TOKEN 就自動使用
import os
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None
if token:
    os.environ["HF_TOKEN"] = token
    print("HF_TOKEN 已載入")
else:
    print("未設定 HF_TOKEN；兩個 public repo 仍可評測，只是下載限速較低")

# 先完整下載兩個 public repo；網路或權限問題會在啟動昂貴的 vLLM engine 前被抓到。
from huggingface_hub import snapshot_download
MODEL_LOCAL = snapshot_download(repo_id=MODEL)
ADAPTER_LOCAL = snapshot_download(repo_id=ADAPTER)
assert pathlib.Path(MODEL_LOCAL, "config.json").exists()
assert pathlib.Path(ADAPTER_LOCAL, "adapter_config.json").exists()
print("Base model 已下載：", MODEL_LOCAL)
print("LoRA adapter 已下載：", ADAPTER_LOCAL)

In [ ]:
# ⑤ 正式跑 base + LoRA 的完整 463 詞評測（最多 6 個批次回合/agent）
# 首次下載與 vLLM compile 可能數分鐘沒有新訊息，請勿中斷。
import pathlib
import subprocess
import sys
import time

pathlib.Path(RESULT_DIR).mkdir(parents=True, exist_ok=True)
log_path = pathlib.Path(RESULT_DIR) / "full_463_eval.log"
cmd = [
    sys.executable, "eval/run_eval.py",
    "--model", MODEL_LOCAL,
    "--adapter", ADAPTER_LOCAL,
    "--label-base", "qwen2.5-1.5b-instruct-base",
    "--backend", "vllm",
    "--split", "eval_full",
    "--n", str(N),
    "--seed", str(SEED),
    "--out", "results/full_463_report.md",
]
print(">>>", " ".join(cmd), flush=True)
eval_started = time.monotonic()
with log_path.open("w", encoding="utf-8") as log:
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
        log.write(line)
    rc = proc.wait()
assert rc == 0, f"評測失敗（returncode={rc}）；不要釋放 runtime，完整 log：{log_path}"
eval_minutes = (time.monotonic() - eval_started) / 60
print(f"完整 463 詞評測完成 ✅，模型推理耗時 {eval_minutes:.1f} 分鐘")

In [ ]:
# ⑥ 驗證並保存報告；完成後才釋放 Colab runtime
import json
import pathlib
import shutil
import subprocess
import sys

report = pathlib.Path("results/full_463_report.md")
payload = pathlib.Path("results/full_463_report.json")
assert report.exists() and payload.exists(), "缺少評測輸出"
data = json.loads(payload.read_text(encoding="utf-8"))
assert data["meta"] == {"n": 463, "seed": 42, "split": "eval_full"}
assert len(data["rows"]) == 2, "完整評測應同時包含 base 與 LoRA 兩列"
assert all(row["n"] == 463 for row in data["rows"].values())

subprocess.run([
    sys.executable, "eval/analyze_full_463.py", "--input", str(payload)
], check=True)
analysis_md = pathlib.Path("results/full_463_analysis.md")
analysis_json = pathlib.Path("results/full_463_analysis.json")
assert analysis_md.exists() and analysis_json.exists()

for artifact in (report, payload, analysis_md, analysis_json):
    shutil.copy2(artifact, pathlib.Path(RESULT_DIR) / artifact.name)
transcripts = pathlib.Path("results/transcripts")
if transcripts.exists():
    shutil.copytree(transcripts, pathlib.Path(RESULT_DIR) / "transcripts", dirs_exist_ok=True)

print(report.read_text(encoding="utf-8"))
print(f"\n已保存到：{RESULT_DIR}")
print("請把 full_463_report.md 與 full_463_report.json 下載或同步回專案 results/。")

from google.colab import runtime
runtime.unassign()